In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append("../..")

from src.configs.settings import Settings
from src.filtration import filter_time_series
from src.ts_features import extract_features_for_groups

In [ ]:
data_dir = Path("../../data")

settings = Settings()

In [ ]:
mirrors_ts_df = pd.read_csv(data_dir / "processed" / "mirrors_ts.csv")

mirrors_products_characteristics_df = pd.read_csv(
    data_dir / "processed" / "mirrors_products_characteristics.csv"
)

In [ ]:
mirrors_ts_df["date"] = pd.to_datetime(mirrors_ts_df["Месяц"])
drop_columns = ["INN", "Месяц", "Выручка FBO", "Выручка FBS", "Выручка", "Продажи FBO"]

mirrors_ts_df = mirrors_ts_df.drop(columns=drop_columns)
mirrors_ts_df = mirrors_ts_df.drop_duplicates()

rename_config = {"NM_ID": "article", "Продажи FBS": "sales"}
mirrors_ts_df = mirrors_ts_df.rename(columns=rename_config)

In [ ]:
int_cols_rename = {
    "options_Габариты__Высота предмета": "height",
    "options_Габариты__Ширина предмета": "width",
    "options_Габариты__Вес без упаковки (кг)": "weight",
}

mirrors_products_characteristics_df = mirrors_products_characteristics_df.drop(columns=["subj_root_name"])
mirrors_products_characteristics_df = mirrors_products_characteristics_df.rename(columns=int_cols_rename)

int_cols = list(int_cols_rename.values())
mirrors_products_characteristics_df[int_cols] = mirrors_products_characteristics_df[int_cols].fillna(
    mirrors_products_characteristics_df[int_cols].median()
)

In [ ]:
filtration_result = filter_time_series(mirrors_ts_df, settings.filtration)
filtered_mirrors_ts_df = filtration_result.df
filtered_mirrors_ts_df.to_csv(data_dir / "processed" / "filtered_mirrors_ts.csv", index=False)
print(f"Осталось {len(filtered_mirrors_ts_df) / len(mirrors_ts_df) * 100:.2f}% строк")
print(f"Отфильтровано панелей: {filtration_result.total_dropped}")
print(f"По шагам: {filtration_result.summary()}")

In [ ]:
import random

nm_id = random.choice(filtered_mirrors_ts_df["article"].unique()).item()
product_df = (
    filtered_mirrors_ts_df[filtered_mirrors_ts_df["article"].eq(nm_id)]
    .sort_values(by=["date"])
    .reset_index(drop=True)
)
product_df["sales"].plot()

In [ ]:
features_mirrors_ts_df = extract_features_for_groups(
    filtered_mirrors_ts_df, group_col="article", value_col="sales"
)
features_mirrors_ts_df = features_mirrors_ts_df.reset_index()

# Add Meta

In [ ]:
features_mirrors_df = pd.merge(
    left=features_mirrors_ts_df,
    right=mirrors_products_characteristics_df,
    how="inner",
    on="article",
)

In [ ]:
features_mirrors_df.to_csv(
    data_dir / "processed" / "features_mirrors.csv", index=False
)